In [23]:
import pathlib
import pandas as pd
import os
import json

def main(input_file, output_file):

    base_dir = pathlib.Path().resolve().parent
    
    multiwoz_path = base_dir / f"multiwoz/data/MultiWOZ_2.2/{input_file}"
    
    if not multiwoz_path.exists():
        raise FileNotFoundError(f"Direktori input tidak ditemukan: {multiwoz_path}")
        
    list_file = [os.path.join(multiwoz_path, f) for f in os.listdir(multiwoz_path) if f.endswith(".json")]
    mwoz_raws = []
    
    for i in list_file:
        print(f"✓ Found file: {i}")
        with open(i, "r", encoding="utf-8") as f:
            multiwoz_raw_file = json.load(f)
            mwoz_raws.extend(multiwoz_raw_file)
        
    # Flatten ke struktur baris
    mwoz_rows = []
    for row in mwoz_raws:
        dialog_id = row['dialogue_id']
        for turn in row['turns']:
            role = "user" if turn['speaker'].lower() == "user" else "agent"
            mwoz_rows.append({
                "room_id": dialog_id,
                "message_id": turn['turn_id'],
                "role": role,
                "message": turn['utterance']
            })

    df_mwoz = pd.DataFrame(mwoz_rows)
    
    # --- PROSES SAMPLING 500 DIALOG UTUH ---
    # 1. Ekstrak seluruh ID unik dari kolom room_id
    unique_rooms = pd.Series(df_mwoz['room_id'].unique())
    
    # 2. Lakukan sampling jika jumlah dialog memenuhi syarat
    if len(unique_rooms) >= 500:
        # Mengambil 500 ID secara acak dengan seed tetap (42) agar reproduksibel
        sampled_rooms = unique_rooms.sample(n=500, random_state=42)
        
        # Saring DataFrame utama: Hanya simpan data yang room_id nya ada di list sampled_rooms
        df_mwoz = df_mwoz[df_mwoz['room_id'].isin(sampled_rooms)].reset_index(drop=True)
    else:
        print(f"⚠️ Peringatan: Jumlah room_id unik ({len(unique_rooms)}) kurang dari 500. Sampling dibatalkan.")
    # ----------------------------------------

    # Menampilkan statistik hasil akhir
    total_dialogues = df_mwoz['room_id'].nunique()
    total_utterances = len(df_mwoz)
    print(f"✓ MultiWOZ (Sampled): {total_dialogues:,} dan utuh, total {total_utterances:,} utterances")

    # Penyimpanan ke folder target
    output_dir = base_dir / 'generate_dataset' / 'multiwoz'
    output_dir.mkdir(parents=True, exist_ok=True)
    
    output_file_path = output_dir / output_file

    # Simpan dengan format JSON Lines (.jsonl)
    df_mwoz.to_json(output_file_path)
    print(f"✓ Berhasil menyimpan dataset sampel ke: {output_file_path}")

# Eksekusi fungsi
main("test", "valid_test_dialogue.json")

✓ Found file: /home/qiscus/Coresearch_ITB_Qiscus/repo_alan/SLM Summarization/multiwoz/data/MultiWOZ_2.2/test/dialogues_001.json
✓ Found file: /home/qiscus/Coresearch_ITB_Qiscus/repo_alan/SLM Summarization/multiwoz/data/MultiWOZ_2.2/test/dialogues_002.json
✓ MultiWOZ (Sampled): 500 dan utuh, total 7,354 utterances
✓ Berhasil menyimpan dataset sampel ke: /home/qiscus/Coresearch_ITB_Qiscus/repo_alan/SLM Summarization/generate_dataset/multiwoz/valid_test_dialogue.json


✓ Found file: /home/qiscus/Coresearch_ITB_Qiscus/repo_alan/SLM Summarization/multiwoz/data/MultiWOZ_2.2/test/dialogues_001.json
✓ Found file: /home/qiscus/Coresearch_ITB_Qiscus/repo_alan/SLM Summarization/multiwoz/data/MultiWOZ_2.2/test/dialogues_002.json
✓ MultiWOZ: 36 dialogues, 14,744 utterances
room_id                                            MUL0484.json
message_id                                                    0
role                                                       user
message       I need train reservations from norwich to camb...
Name: 0, dtype: object
